In [3]:
from transformers import MarianMTModel, MarianTokenizer

model_name = "Helsinki-NLP/opus-mt-en-fr"

tokenizer = MarianTokenizer.from_pretrained(model_name)
model = MarianMTModel.from_pretrained(model_name)

print("Model loaded successfully")


Model loaded successfully


Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model.safetensors:   0%|          | 0.00/301M [00:00<?, ?B/s]

In [4]:
text = "Machine learning is very powerful."

inputs = tokenizer(text, return_tensors="pt")

translated = model.generate(**inputs)

output = tokenizer.decode(translated[0], skip_special_tokens=True)

print("English:", text)
print("French:", output)


English: Machine learning is very powerful.
French: L'apprentissage automatique est très puissant.


In [5]:
from datasets import load_dataset

dataset = load_dataset("opus100","en-fr")

print(dataset['train'][0])

README.md: 0.00B [00:00, ?B/s]

C:\Users\23adsb49\AppData\Roaming\Python\Python39\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\23adsb49\.cache\huggingface\hub\datasets--opus100. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP d

test-00000-of-00001.parquet:   0%|          | 0.00/327k [00:00<?, ?B/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


train-00000-of-00001.parquet:   0%|          | 0.00/142M [00:00<?, ?B/s]

...

validation-00000-of-00001.parquet:   0%|          | 0.00/334k [00:00<?, ?B/s]

Generating test split:   0%|          | 0/2000 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/1000000 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/2000 [00:00<?, ? examples/s]

{'translation': {'en': 'The time now is 05:08 .', 'fr': 'The time now is 05:05 .'}}


In [6]:
sentence = dataset['train'][0]['translation']['en']

inputs = tokenizer(sentence, return_tensors="pt")

translated = model.generate(**inputs)

output = tokenizer.decode(translated[0], skip_special_tokens=True)

print("English:", sentence)
print("French:", output)

English: The time now is 05:08 .
French: Le temps est maintenant 05:08 .


In [7]:
sentences = [
    "Artificial intelligence is changing the world.",
    "Natural language processing is interesting.",
    "Transformers are powerful models."
]

for s in sentences:
    inputs = tokenizer(s, return_tensors="pt")
    translated = model.generate(**inputs)
    result = tokenizer.decode(translated[0], skip_special_tokens=True)

    print("\nEnglish:", s)
    print("French:", result)


English: Artificial intelligence is changing the world.
French: L'intelligence artificielle change le monde.

English: Natural language processing is interesting.
French: Le traitement du langage naturel est intéressant.

English: Transformers are powerful models.
French: Les transformateurs sont des modèles puissants.


In [8]:
for i in range(3):
    en = dataset['train'][i]['translation']['en']

    inputs = tokenizer(en, return_tensors="pt")
    translated = model.generate(**inputs)

    fr = tokenizer.decode(translated[0], skip_special_tokens=True)

    print("\nEnglish:", en)
    print("Predicted French:", fr)


English: The time now is 05:08 .
Predicted French: Le temps est maintenant 05:08 .

English: This Regulation shall enter into force on the seventh day following its publication in the Official Journal of the European Union.
Predicted French: Le présent règlement entre en vigueur le septième jour suivant celui de sa publication au Journal officiel de l'Union européenne.

English: Hello, what's that?
Predicted French: Bonjour, c'est quoi ?


In [9]:
# ---------------------------------------
# Install libraries (run once if needed)
# ---------------------------------------
# pip install transformers datasets sentencepiece sacrebleu torch

# ---------------------------------------
# Import Libraries
# ---------------------------------------

from datasets import load_dataset
from transformers import MarianMTModel, MarianTokenizer
import sacrebleu

# ---------------------------------------
# 1. Load the English-French Dataset
# ---------------------------------------

dataset = load_dataset("opus100", "en-fr")

print("Dataset Loaded Successfully")
print(dataset)

# Show one example
sample = dataset['train'][0]['translation']
print("\nSample Data:")
print("English:", sample['en'])
print("French :", sample['fr'])

# ---------------------------------------
# 2. Load Transformer Translation Model
# ---------------------------------------

model_name = "Helsinki-NLP/opus-mt-en-fr"

tokenizer = MarianTokenizer.from_pretrained(model_name)
model = MarianMTModel.from_pretrained(model_name)

print("\nModel Loaded Successfully")

# ---------------------------------------
# 3. Translation Function
# ---------------------------------------

def translate_text(text):

    tokens = tokenizer(text, return_tensors="pt", padding=True, truncation=True)

    translated = model.generate(**tokens)

    result = tokenizer.decode(translated[0], skip_special_tokens=True)

    return result

# ---------------------------------------
# 4. Translate Custom Sentences
# ---------------------------------------

sentences = [
    "Artificial intelligence is transforming the world.",
    "Machine learning is very powerful.",
    "Natural language processing is interesting."
]

print("\nCustom Sentence Translation:")

for s in sentences:
    translated = translate_text(s)

    print("\nEnglish :", s)
    print("French  :", translated)

# ---------------------------------------
# 5. Translate Sentences from Dataset
# ---------------------------------------

print("\nDataset Translation Examples:")

references = []
predictions = []

for i in range(10):

    en = dataset['test'][i]['translation']['en']
    fr_true = dataset['test'][i]['translation']['fr']

    fr_pred = translate_text(en)

    print("\nEnglish :", en)
    print("Actual French :", fr_true)
    print("Predicted French :", fr_pred)

    references.append([fr_true])
    predictions.append(fr_pred)

# ---------------------------------------
# 6. BLEU Score Evaluation
# ---------------------------------------

bleu = sacrebleu.corpus_bleu(predictions, references)

print("\nBLEU Score:", bleu.score)

Dataset Loaded Successfully
DatasetDict({
    test: Dataset({
        features: ['translation'],
        num_rows: 2000
    })
    train: Dataset({
        features: ['translation'],
        num_rows: 1000000
    })
    validation: Dataset({
        features: ['translation'],
        num_rows: 2000
    })
})

Sample Data:
English: The time now is 05:08 .
French : The time now is 05:05 .

Model Loaded Successfully

Custom Sentence Translation:

English : Artificial intelligence is transforming the world.
French  : L'intelligence artificielle transforme le monde.

English : Machine learning is very powerful.
French  : L'apprentissage automatique est très puissant.

English : Natural language processing is interesting.
French  : Le traitement du langage naturel est intéressant.

Dataset Translation Examples:

English : - You were at a bus stop kissing him!
Actual French : - Vous étiez en train de vous embrasser à l'arrêt de bus!
Predicted French : - Tu étais à un arrêt de bus pour l'embra